In [0]:
import sys
import os
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from modules.utils.date import get_target_yyyymm
from modules.data.file_download import download_file_from_zip

In [0]:
# Download and load to volume bike trips history, given a city and target past month.
# 1. Construct file paths based on city/system and target month.
# 2. If file already exists, mark job as complete for downstream.
# 3. If not, attempt to download the file from the city's list of trip URLs and upload to target path.
# 4. If successful, mark job as ready for downstream tasks.
months_ago = int(dbutils.widgets.get("months_ago"))
target_month = get_target_yyyymm(months_ago=months_ago)
city = dbutils.widgets.get("city")
city_dict = json.loads(dbutils.widgets.get(city))

file_name: str = f"{target_month.replace("-","")}-{city_dict["system"]}-tripdata.csv"
dir_path: str = f"/Volumes/bikes/00_landing/data_sources/{city_dict["system"]}/trips/{target_month}"
local_path: str = f"{dir_path}/{file_name}"

try:
    dbutils.fs.ls(local_path)

    dbutils.jobs.taskValues.set(key='continue_downstream', value="No")
    print('File already downloaded, aborting downstream tasks')
except:
    try:
        for url in city_dict["trips_urls"]:
            url = url.replace("($date)",target_month.replace("-",""))
            if download_file_from_zip(url=url, file_name=file_name, dir_path=dir_path, local_path=local_path):
                dbutils.jobs.taskValues.set(key='continue_downstream', value="Yes")
                print('File succesfully uploaded in current run')
                break
            dbutils.jobs.taskValues.set(key='continue_downstream', value="No")
    except Exception as e:
        print(f"Error while downloading file: {str(e)}")
        dbutils.jobs.taskValues.set(key='continue_downstream', value="No")